In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

<details>
<summary><b>Versions des packages</b></summary>

Le code de cette page a été développé avec les dépendances suivantes.
Nous recommandons d'utiliser ces versions ou des versions plus récentes.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
En plus de [visualiser les instructions d'un circuit](/guides/visualize-circuits), tu pourrais vouloir visualiser l'ordonnancement d'un circuit en utilisant la méthode Qiskit [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer). Cette visualisation peut t'aider, par exemple, à repérer rapidement les temps d'inactivité sur les qubits. Cependant, cette méthode ne retourne pas de résultats précis pour les circuits dynamiques. Pour visualiser l'ordonnancement d'un circuit dynamique, utilise la méthode `draw_circuit_schedule_timing`, décrite dans la section [Support de Qiskit Runtime](#qr-support).

## Exemples

Pour visualiser un programme de circuit ordonné, tu peux appeler cette fonction avec un ensemble d'arguments de contrôle. La plupart de l'apparence de l'image de sortie peut être modifiée via une feuille de style, mais ce n'est pas obligatoire.

### Afficher avec la feuille de style par défaut

In [1]:
from qiskit import QuantumCircuit
from qiskit.visualization.timeline import draw
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

backend = GenericBackendV2(5)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

draw(isa_circuit, target=backend.target)

<Image src="../docs/images/guides/visualize-circuit-timing/extracted-outputs/5b21b8c1-0.svg" alt="Output of the previous code cell" />

![Sortie de la cellule de code précédente](../docs/images/guides/visualize-circuit-timing/extracted-outputs/5b21b8c1-0.svg)

### Afficher avec une feuille de style adaptée au débogage de programmes

In [2]:
from qiskit import QuantumCircuit
from qiskit.visualization.timeline import draw, IQXDebugging
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

backend = GenericBackendV2(5)
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)
draw(isa_circuit, style=IQXDebugging(), target=backend.target)

<Image src="../docs/images/guides/visualize-circuit-timing/extracted-outputs/907dc46c-0.svg" alt="Output of the previous code cell" />

![Sortie de la cellule de code précédente](../docs/images/guides/visualize-circuit-timing/extracted-outputs/907dc46c-0.svg)

Tu peux créer des fonctions de génération ou de mise en page personnalisées et mettre à jour une feuille de style existante avec ces fonctions. De cette façon, tu peux contrôler la majeure partie de l'apparence de l'image de sortie sans modifier la base de code du dessinateur de circuits ordonnancés. Consulte la référence API [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer) pour plus d'exemples.
<span id="qr-support"></span>
## Support de Qiskit Runtime
Bien que le dessinateur de timeline intégré à Qiskit soit utile pour les circuits statiques, il pourrait ne pas refléter avec précision le timing des [circuits dynamiques](/guides/classical-feedforward-and-control-flow), en raison d'opérations implicites telles que la diffusion et la détermination de branches. Dans le cadre de la prise en charge des circuits dynamiques, Qiskit Runtime retourne les informations précises de timing du circuit dans les résultats du job lorsqu'on le demande.

> **Note:** - Il s'agit d'une fonction expérimentale. Elle est en version préliminaire et est donc susceptible d'évoluer.
> - Cette fonction s'applique uniquement aux jobs du Sampler de Qiskit Runtime.
> - Bien que le temps total du circuit soit retourné dans les métadonnées de « compilation », ce n'est PAS le temps utilisé pour la facturation (temps quantique).
### Activer la récupération des données de timing
Pour activer la récupération des données de timing, définis le flag expérimental `scheduler_timing` à `True` lors de l'exécution du job primitif.